# Chunking the EU AI Act (KAN-92, KAN-93)
Gözde Nur Özdemir · ELIZA · Sprint 2

**Goal:** split the AI Act into chunks, one chunk per provision in our provision list, and add metadata to each chunk.

**How to run:** open this notebook from the `src/` folder of the repository. It reads `config/rag.yaml` and the files in `data/`, and writes `data/chunks.csv` and `data/chunks.jsonl`.

**Steps**
1. Load the HTML of the AI Act
2. Load the provision list.
3. Read the HTML into small units (paragraphs, points)
4. Build one chunk per provision
5. Split long chunks and link each chunk to the one before and after it
6. Save and look at the chunks

In [1]:
# Install libraries
%pip -q install beautifulsoup4 pandas pyyaml

import csv, hashlib, io, json, os, re
import pandas as pd
import requests
import yaml
from bs4 import BeautifulSoup

CONFIG = yaml.safe_load(open("../config/rag.yaml"))["chunking"]   # the chunking settings
CONFIG

Note: you may need to restart the kernel to use updated packages.


{'strategy': 'structure',
 'source_file': 'CELEX_02024R1689-20260727_EN.html',
 'version_date': '27/07/2026',
 'celex': '02024R1689-20260727',
 'max_chars': 1500}

## 1. Load the HTML
We use the consolidated text of 27/07/2026 from EUR-Lex (design decision D8).
The file is in `data/`. If it is missing, we download it from the link. Then we check its SHA-256 against `data/manifest.csv`, so that we always use the same version.

In [2]:
HTML_FILE = "../data/" + CONFIG["source_file"]
HTML_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:" + CONFIG["celex"]

if not os.path.exists(HTML_FILE):
    r = requests.get(HTML_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status()
    open(HTML_FILE, "wb").write(r.content)

# check that the file is the version written in data/manifest.csv (SRC-02)
manifest = {row["file"]: row["sha256"] for row in csv.DictReader(open("../data/manifest.csv", encoding="utf-8"))}
sha = hashlib.sha256(open(HTML_FILE, "rb").read()).hexdigest()
assert sha == manifest["data/" + CONFIG["source_file"]], "This HTML is not the version in data/manifest.csv"

html = open(HTML_FILE, encoding="utf-8").read()
soup = BeautifulSoup(html, "html.parser")
print("HTML size:", len(html), "characters")
print("SHA-256 matches data/manifest.csv:", sha)

HTML size: 850466 characters
SHA-256 matches data/manifest.csv: e7184c2b0a8037fb7e8ab8a98ab70832513579e841e520363bcca6a9ced0a2ea


**Comment:** The HTML file has about 850,000 characters. Its SHA-256 is the same as in `data/manifest.csv` (SRC-02), so we use exactly the version of the law we planned to use.

## 2. Load the provision list
`provision-list.csv` says which provisions are in our scope (C-01). Each row becomes one chunk.

In [3]:
LIST_FILE = "../data/provision-list.csv"
LIST_URL = "https://raw.githubusercontent.com/ELIZA-DI502/ELIZA_DI502_ORGANIZATION/main/data/provision-list.csv"

if not os.path.exists(LIST_FILE):
    open(LIST_FILE, "wb").write(requests.get(LIST_URL, timeout=60).content)

provisions = list(csv.DictReader(open(LIST_FILE, encoding="utf-8")))
print(len(provisions), "provisions, for example:", provisions[0]["reference"])

103 provisions, for example: Annex III, point 3(a)


**Comment:** There are 103 provisions in our scope. The first one is Annex III, point 3(a), which is about AI in education.

## 3. Read the HTML into units
The HTML marks every part of the law:
- an Article is `<div id="art_6">`
- a numbered paragraph starts with `<span class="no-parag">3. </span>`
- a point like (a) is a list box (`grid-container`)

We walk through each Article and the Annexes and save every paragraph, subparagraph, point and sub-point as one **unit**.

In [4]:
units = []   # every small part of the law: paragraphs, subparagraphs, points

def text(el):
    """Returns the text of an HTML element with single spaces."""
    return re.sub(r"\s+", " ", el.get_text(" ")).strip()

def add(info, place, t):
    """Saves one unit: where it is in the law (info, place) and its text t."""
    units.append({**info, **place, "text": t})

def read_box(box, info, place, depth):
    """Reads a list box: '3.' (Annex area), '(a)' (point) or '(i)' (sub-point), then the boxes inside it.
    depth is 0 for the outer box, 1 for a box inside it, and so on."""
    label = text(box.find("div", class_="grid-list-column-1"))
    content = box.find("div", class_="grid-list-column-2")
    levels = ["point", "subpoint"] if info["kind"] == "article" else ["para", "point", "subpoint"]
    place = {**place, "sub": place["sub"] or 1}
    level = levels[min(depth, len(levels) - 1)]   # deeper boxes count as sub-points
    if level == "para":
        place.update(para=label.rstrip("."), point=None, subpoint=None)
    elif level == "point":
        place.update(point=label.strip("()"), subpoint=None)
    else:
        place["subpoint"] = label.strip("()")
    lines = [text(c) for c in content.find_all(recursive=False) if "grid-container" not in c.get("class", []) and text(c)]
    add(info, place, label + " " + "\n".join(lines))
    for small in content.find_all("div", class_="grid-container", recursive=False):
        read_box(small, info, place, depth + 1)

def walk(parent, info, place):
    """Goes through the children of an HTML element in order and saves every paragraph, subparagraph and point."""
    for el in parent.find_all(recursive=False):
        cls = el.get("class", [])
        number = el.find("span", class_="no-parag", recursive=False)
        if number:                                          # numbered paragraph "3."
            place.update(para=text(number).rstrip("."), sub=0, point=None, subpoint=None)
            start = len(units)
            walk(el, info, place)
            units[start]["text"] = place["para"] + ". " + units[start]["text"]
        elif el.name == "p" and "norm" in cls and text(el):  # a new subparagraph
            place.update(sub=place["sub"] + 1, point=None, subpoint=None)
            add(info, place, text(el))
        elif "inline-element" in cls and not el.find(["p", "div"], recursive=False):
            place["sub"] = 1                                # paragraph that is one block of text
            add(info, place, text(el))
        elif "inline-element" in cls:
            walk(el, info, place)
        elif "grid-container" in cls:                        # a point
            read_box(el, info, place, 0)

for div in soup.find_all("div", id=re.compile(r"^(art_\d+[a-z]?|anx_[IVX]+)$")):
    kind = "article" if div["id"].startswith("art") else "annex"
    title = div.find("p", class_=["stitle-article-norm", "title-annex-2"])
    chapter = div.find_parent(id=re.compile(r"^cpt_"))
    info = {"kind": kind, "num": div["id"].split("_")[1], "title": text(title) if title else "",
            "chapter": chapter["id"][4:].split(".")[0] if chapter and kind == "article" else ""}
    walk(div, info, {"para": None, "sub": 0, "point": None, "subpoint": None})

print(len(units), "units read from the HTML")

pd.DataFrame(units)[["kind", "num", "para", "sub", "point", "text"]].head(10)   # the first 10 units

1377 units read from the HTML


,kind,num,para,sub,point,text
0,article,1,1,1,NaN,1. The purpose of this Regulation is to improv...
1,article,1,2,1,NaN,2. This Regulation lays down:
2,article,1,2,1,a,(a) harmonised rules for the placing on the ma...
3,article,1,2,1,b,(b) prohibitions of certain AI practices;
4,article,1,2,1,c,(c) specific requirements for high-risk AI sys...
5,article,1,2,1,d,(d) harmonised transparency rules for certain ...
6,article,1,2,1,e,(e) harmonised rules for the placing on the ma...
7,article,1,2,1,f,"(f) rules on market monitoring, market surveil..."
8,article,1,2,1,g,"(g) measures to support innovation, with a par..."
9,article,2,1,1,NaN,1. This Regulation applies to:


**Comment:** The HTML gave 1,377 small units. The table shows that Article 1 is split into its paragraphs and its points (a) to (g), so each part of the law is now separate.

## 4. One chunk per provision
Each row of the provision list has a reference like `Article 6(3), second subparagraph, point (a)`.
We turn it into numbers (`num=6, para=3, sub=2, point=a`) and collect the units that match.

Each chunk gets:
- **text**: the provision itself (this is what an answer quotes)
- **embed_text**: heading + the sentence that introduces the point + text (this is what we search), so that a short point like "(a) …narrow procedural task" still makes sense
- **metadata**: reference, title, chapter, path, priority, version date, amendment

In [5]:
NUMBER = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5}

def parse_reference(ref):
    """'Article 6(3), second subparagraph, point (a)' -> num 6, para 3, sub 2, point a"""
    if ref.startswith("Annex"):                                    # Annex III, point 3(a)
        _, num, _, para, point = re.findall(r"\w+", ref)
        return {"kind": "annex", "num": num, "para": para, "sub": None, "point": point, "subpoint": None}
    num, para = re.match(r"Article (\w+)\(?(\w*)", ref).groups()   # Article 6(3)
    sub = re.search(r"(\w+) (?:sub)?paragraph", ref)               # second subparagraph
    points = re.findall(r"\((\w+)\)", ref.partition("point")[2]) + [None, None]   # point (c)(i)
    return {"kind": "article", "num": num, "para": para or None, "sub": NUMBER[sub[1]] if sub else None,
            "point": points[0], "subpoint": points[1]}

chunks = []
for row in provisions:
    ref = parse_reference(row["reference"])
    parts = [u for u in units if all(ref[k] in (None, u[k]) for k in ref)]   # the units of this provision
    if not parts:
        print("NOT FOUND:", row["reference"])
        continue
    u = parts[0]
    text = "\n".join(p["text"] for p in parts)
    # heading, e.g. "Article 6 – Classification rules for high-risk AI systems – paragraph 3:"
    name = ("Article " if u["kind"] == "article" else "Annex ") + u["num"]
    level = " – paragraph " if u["kind"] == "article" else " – point "
    heading = name + " – " + u["title"] + (level + ref["para"] if ref["para"] else "") + ":"
    # lead-in for a point, e.g. "The first subparagraph shall apply where ...:"
    lead = {**u, "point": u["point"] if ref["subpoint"] else None, "subpoint": None}
    intro = next((x["text"] for x in units if ref["point"] and x["text"].endswith(":")
                  and all(x[k] == lead[k] for k in ref)), "")
    chunks.append({
        "chunk_id": row["provision_id"], "ref": row["reference"],
        "parent": row["reference"].split(",")[0] if u["kind"] == "article" else row["reference"].split("(")[0],
        "kind": u["kind"], "number": u["num"], "title": u["title"], "chapter": u["chapter"],
        "priority": row["priority"], "area": row["area"], "amended_by": row["amended_by_2026_1744"],
        "version_date": CONFIG["version_date"], "celex": CONFIG["celex"], "language": "en",
        "source": "data/" + CONFIG["source_file"],
        "text": text, "heading": heading, "intro": intro,
        "embed_text": "\n".join(x for x in (heading, intro, text) if x),
        "order": units.index(u),
    })

chunks.sort(key=lambda c: c["order"])          # put them in the order of the law
print(len(chunks), "of", len(provisions), "provisions found")

103 of 103 provisions found


**Comment:** All 103 provisions were found in the HTML, so nothing is missing. Each provision is now one chunk with its text, a heading and metadata.

## 5. Split long chunks, link neighbours
- A provision longer than **1,500 characters** (`max_chars` in `config/rag.yaml`) is split at its points (or sentences). Each part keeps the heading. (RAGTurk splits long sections the same way.)
- Each chunk gets `prev_id` and `next_id`: the chunk before and after it in the same Article, so the retriever can add neighbours later.

In [6]:
MAX_CHARS = CONFIG["max_chars"]   # 1500, from config/rag.yaml

def split_text(text, limit):
    """Splits a text at its lines into pieces of at most `limit` characters (a line is never cut)."""
    pieces, current = [], ""
    for line in text.split("\n"):
        if current and len(current) + len(line) > limit:
            pieces.append(current)
            current = line
        else:
            current = (current + "\n" + line) if current else line
    return pieces + [current]

final = []
for c in chunks:
    if len(c["text"]) <= MAX_CHARS:
        final.append(c)
        continue
    pieces = split_text(c["text"], MAX_CHARS)
    for n, piece in enumerate(pieces, 1):
        final.append({**c, "chunk_id": f"{c['chunk_id']}#{n}", "part": f"{n}/{len(pieces)}", "text": piece,
                      "embed_text": "\n".join(x for x in (c["heading"], c["intro"], piece) if x)})

for i, c in enumerate(final):
    same = lambda other: other["kind"] == c["kind"] and other["number"] == c["number"]
    c["prev_id"] = final[i - 1]["chunk_id"] if i > 0 and same(final[i - 1]) else ""
    c["next_id"] = final[i + 1]["chunk_id"] if i + 1 < len(final) and same(final[i + 1]) else ""
    c["words"] = len(c["text"].split())

print(len(final), "chunks; split:", sorted({c["chunk_id"].split("#")[0] for c in final if "#" in c["chunk_id"]}))

105 chunks; split: ['ART13-3', 'ART4A-1']


**Comment:** Only two provisions were longer than 1,500 characters: Article 13(3) and Article 4a(1). Each was split into two parts, so we have 105 chunks in total.

## 6. Save and look at the chunks

In [7]:
with open("../data/chunks.jsonl", "w", encoding="utf-8") as f:
    for c in final:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
df = pd.DataFrame(final)
df.drop(columns=["embed_text", "heading", "intro", "order"]).to_csv("../data/chunks.csv", index=False)

print("saved data/chunks.jsonl and data/chunks.csv")
df["start"] = df["text"].str[:60] + "..."   # how each chunk begins
df[["chunk_id", "ref", "start", "prev_id", "next_id", "words"]].head(10)   # the first 10 chunks

saved data/chunks.jsonl and data/chunks.csv


,chunk_id,ref,start,prev_id,next_id,words
0,ART2-1,Article 2(1),1. This Regulation applies to:\n(a) providers ...,,ART2-2,148
1,ART2-2,Article 2(2),2. For AI systems classified as high-risk AI s...,ART2-1,ART2-3,71
2,ART2-3,Article 2(3),3. This Regulation does not apply to areas out...,ART2-2,ART2-4,144
3,ART2-4,Article 2(4),4. This Regulation applies neither to public a...,ART2-3,ART2-5,83
4,ART2-5,Article 2(5),5. This Regulation shall not affect the applic...,ART2-4,ART2-6,29
5,ART2-6,Article 2(6),6. This Regulation does not apply to AI system...,ART2-5,ART2-7,30
6,ART2-7,Article 2(7),7. Union law on the protection of personal dat...,ART2-6,ART2-8,59
7,ART2-8,Article 2(8),8. This Regulation does not apply to any resea...,ART2-7,ART2-9,54
8,ART2-9,Article 2(9),9. This Regulation is without prejudice to the...,ART2-8,ART2-10,23
9,ART2-10,Article 2(10),10. This Regulation does not apply to obligati...,ART2-9,ART2-11,26


**Comment:** Each chunk in the table is one paragraph of Article 2, and it points to the chunk before and after it. The first chunk has no prev_id, because it is the start of the Article.

In [8]:
# Look at one chunk. Change the ID to see another one: ANX3-3-b, ART113-3-c-i, ART4A-1#1
c = next(c for c in final if c["chunk_id"] == "ART6-3-a")
print("TEXT:\n" + c["text"] + "\n")
print("EMBED_TEXT:\n" + c["embed_text"] + "\n")
for k in ["ref", "parent", "prev_id", "next_id", "chapter", "priority", "version_date", "amended_by"]:
    print(f"{k}: {c[k]}")

TEXT:
(a) the AI system is intended to perform a narrow procedural task;

EMBED_TEXT:
Article 6 – Classification rules for high-risk AI systems – paragraph 3:
The first subparagraph shall apply where any of the following conditions is fulfilled:
(a) the AI system is intended to perform a narrow procedural task;

ref: Article 6(3), second subparagraph, point (a)
parent: Article 6(3)
prev_id: ART6-3-1
next_id: ART6-3-b
chapter: III
priority: core
version_date: 27/07/2026
amended_by: 


**Comment:** Point (a) alone is very short and does not say what it belongs to. The embed text adds the Article heading and the sentence before the point, so the search can understand it.